In [0]:
WITH segmented_users AS (
    SELECT
        countryCode,
        identifierHash,

        CASE
            WHEN CAST(dayssincelastlogin AS INT) <= 30
                THEN 'Active'
            WHEN CAST(dayssincelastlogin AS INT) <= 90
                THEN 'Cooling'
            WHEN CAST(dayssincelastlogin AS INT) <= 365
                THEN 'At Risk'
            ELSE 'Dormant'
        END AS activity_segment

    FROM ecommerce_fashion.gold.comprehensive_table
    WHERE LOWER(type) = 'user'
),
segment_counts AS (
    SELECT
        countryCode,
        activity_segment,
        COUNT(DISTINCT identifierHash) AS users
    FROM segmented_users
    GROUP BY countryCode, activity_segment
)
SELECT
    countryCode,
    activity_segment,
    users,

    ROUND(
        100.0 * users / NULLIF(SUM(users) OVER (PARTITION BY countryCode), 0),
        2
    ) AS country_user_pct

FROM segment_counts
ORDER BY countryCode, users DESC;